<a href="https://colab.research.google.com/github/flaviaanbu10-cmd/MyFirstPythonCode/blob/main/medication_safety_copilot_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medication Safety Copilot (Starter Project)

In this starter project, you build a small medication safety tool that looks for possible:

- drug interaction risks
- duplicate therapy
- prescribing cascades
- OTC/supplement concerns

This is a **rule-based foundation**. Later, you can build on it with AI features such as:
- better symptom understanding
- smarter medication matching
- patient summary generation
- caregiver-friendly explanations
- risk scoring


In [ ]:
import pandas as pd

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 1: Create a small sample dataset

This is a simple starter dataset.

Each row is one patient.

Columns:
- `Patient`: patient name
- `Age`: age in years
- `Diagnoses`: known conditions
- `Medications`: prescription medications
- `OTC`: over-the-counter medications
- `Supplements`: vitamins/herbal products
- `Symptoms`: current symptoms
- `Recent_Changes`: recent medication changes


In [ ]:
import pandas as pd

data = pd.DataFrame({
    "Patient": ["Mary", "John", "Rosa", "David"],
    "Age": [78, 72, 81, 69],
    "Diagnoses": [
        ["neuropathy", "hypertension"],
        ["arthritis", "heartburn"],
        ["insomnia", "allergies"],
        ["chronic pain", "hypertension"]
    ],
    "Medications": [
        ["gabapentin", "lisinopril", "furosemide"],
        ["ibuprofen", "naproxen", "omeprazole"],
        ["diphenhydramine", "zolpidem"],
        ["amlodipine", "ibuprofen"]
    ],
    "OTC": [
        [],
        ["ibuprofen"],
        ["diphenhydramine"],
        []
    ],
    "Supplements": [
        ["ginkgo"],
        [],
        ["melatonin"],
        []
    ],
    "Symptoms": [
        ["leg swelling", "dizziness"],
        ["stomach pain"],
        ["daytime sleepiness", "confusion"],
        ["ankle swelling"]
    ],
    "Recent_Changes": [
        ["started furosemide after swelling"],
        ["started naproxen for pain"],
        ["takes diphenhydramine most nights"],
        ["started amlodipine 2 weeks ago"]
    ]
})

data

,Patient,Age,Diagnoses,Medications,OTC,Supplements,Symptoms,Recent_Changes
0,Mary,78,"[neuropathy, hypertension]","[gabapentin, lisinopril, furosemide]",[],[ginkgo],"[leg swelling, dizziness]",[started furosemide after swelling]
1,John,72,"[arthritis, heartburn]","[ibuprofen, naproxen, omeprazole]",[ibuprofen],[],[stomach pain],[started naproxen for pain]
2,Rosa,81,"[insomnia, allergies]","[diphenhydramine, zolpidem]",[diphenhydramine],[melatonin],"[daytime sleepiness, confusion]",[takes diphenhydramine most nights]
3,David,69,"[chronic pain, hypertension]","[amlodipine, ibuprofen]",[],[],[ankle swelling],[started amlodipine 2 weeks ago]


## Step 2: Define your starter knowledge base

These are the first safety rules.

You can add many more later.


In [ ]:
interaction_rules = [
    {
        "meds": {"ibuprofen", "naproxen"},
        "issue": "Possible duplicate NSAID therapy",
        "reason": "Using two NSAIDs together can raise the risk of stomach bleeding and kidney problems."
    },
    {
        "meds": {"diphenhydramine", "zolpidem"},
        "issue": "Sedation / fall risk",
        "reason": "These medicines together may increase sleepiness, confusion, and fall risk."
    },
    {
        "meds": {"ginkgo", "ibuprofen"},
        "issue": "Possible bleeding risk",
        "reason": "Ginkgo with ibuprofen may increase bleeding risk in some patients."
    }
]

duplicate_groups = {
    "NSAIDs": {"ibuprofen", "naproxen", "diclofenac"},
    "Sleep_Aids": {"diphenhydramine", "zolpidem", "melatonin"}
}

cascade_rules = [
    {
        "trigger_med": "gabapentin",
        "symptom": "leg swelling",
        "followup_med": "furosemide",
        "issue": "Possible prescribing cascade",
        "reason": "Gabapentin can sometimes contribute to swelling, and furosemide may have been added to treat the side effect."
    },
    {
        "trigger_med": "amlodipine",
        "symptom": "ankle swelling",
        "followup_med": None,
        "issue": "Possible medication side effect",
        "reason": "Amlodipine can cause ankle swelling, which may later lead to an unnecessary second medicine."
    }
]

## Step 3: Build helper functions

These functions clean the input and check each patient.


In [ ]:
def normalize_list(items):
    return {str(item).strip().lower() for item in items}

def combine_all_meds(row):
    meds = []
    meds.extend(row["Medications"])
    meds.extend(row["OTC"])
    meds.extend(row["Supplements"])
    return normalize_list(meds)

def check_interactions(all_meds):
    findings = []
    for rule in interaction_rules:
        if rule["meds"].issubset(all_meds):
            findings.append({
                "Type": "Interaction",
                "Issue": rule["issue"],
                "Reason": rule["reason"]
            })
    return findings

def check_duplicates(all_meds):
    findings = []
    for group_name, med_group in duplicate_groups.items():
        overlap = all_meds.intersection({m.lower() for m in med_group})
        if len(overlap) >= 2:
            findings.append({
                "Type": "Duplicate Therapy",
                "Issue": f"Multiple medicines in {group_name}",
                "Reason": f"Found these together: {', '.join(sorted(overlap))}"
            })
    return findings

def check_cascades(row, all_meds):
    findings = []
    symptoms = normalize_list(row["Symptoms"])
    recent_changes_text = " ".join(row["Recent_Changes"]).lower()

    for rule in cascade_rules:
        trigger_present = rule["trigger_med"].lower() in all_meds
        symptom_present = rule["symptom"].lower() in symptoms
        followup_present = True if rule["followup_med"] is None else rule["followup_med"].lower() in all_meds

        if trigger_present and symptom_present and followup_present:
            findings.append({
                "Type": "Prescribing Cascade",
                "Issue": rule["issue"],
                "Reason": rule["reason"],
                "Recent_Changes": recent_changes_text
            })
    return findings

## Step 4: Create the patient analyzer

This function runs all safety checks for one patient.


In [ ]:
def analyze_patient(row):
    all_meds = combine_all_meds(row)

    findings = []
    findings.extend(check_interactions(all_meds))
    findings.extend(check_duplicates(all_meds))
    findings.extend(check_cascades(row, all_meds))

    if not findings:
        findings.append({
            "Type": "No major flag found",
            "Issue": "No starter-rule issue detected",
            "Reason": "This does not mean the patient has no risk. It only means no current starter rule was triggered."
        })

    result = pd.DataFrame(findings)
    result.insert(0, "Patient", row["Patient"])
    return result

## Step 5: Run the tool on all patients


In [ ]:
reports = []

for _, row in data.iterrows():
    report = analyze_patient(row)
    reports.append(report)

final_report = pd.concat(reports, ignore_index=True)
final_report

,Patient,Type,Issue,Reason,Recent_Changes
0,Mary,Prescribing Cascade,Possible prescribing cascade,Gabapentin can sometimes contribute to swellin...,started furosemide after swelling
1,John,Interaction,Possible duplicate NSAID therapy,Using two NSAIDs together can raise the risk o...,NaN
2,John,Duplicate Therapy,Multiple medicines in NSAIDs,"Found these together: ibuprofen, naproxen",NaN
3,Rosa,Interaction,Sedation / fall risk,These medicines together may increase sleepine...,NaN
4,Rosa,Duplicate Therapy,Multiple medicines in Sleep_Aids,"Found these together: diphenhydramine, melaton...",NaN
5,David,Prescribing Cascade,Possible medication side effect,"Amlodipine can cause ankle swelling, which may...",started amlodipine 2 weeks ago


## Step 6: Create a simple caregiver-friendly summary


In [ ]:
for patient_name in data["Patient"]:
    patient_rows = final_report[final_report["Patient"] == patient_name]
    print("=" * 60)
    print(f"Patient: {patient_name}")
    for _, item in patient_rows.iterrows():
        print(f"- {item['Type']}: {item['Issue']}")
        print(f"  Why it matters: {item['Reason']}")
    print("  Reminder: Review these findings with a doctor or pharmacist.")
    print()

Patient: Mary
- Prescribing Cascade: Possible prescribing cascade
  Why it matters: Gabapentin can sometimes contribute to swelling, and furosemide may have been added to treat the side effect.
  Reminder: Review these findings with a doctor or pharmacist.

Patient: John
- Interaction: Possible duplicate NSAID therapy
  Why it matters: Using two NSAIDs together can raise the risk of stomach bleeding and kidney problems.
- Duplicate Therapy: Multiple medicines in NSAIDs
  Why it matters: Found these together: ibuprofen, naproxen
  Reminder: Review these findings with a doctor or pharmacist.

Patient: Rosa
- Interaction: Sedation / fall risk
  Why it matters: These medicines together may increase sleepiness, confusion, and fall risk.
- Duplicate Therapy: Multiple medicines in Sleep_Aids
  Why it matters: Found these together: diphenhydramine, melatonin, zolpidem
  Reminder: Review these findings with a doctor or pharmacist.

Patient: David
- Prescribing Cascade: Possible medication side 

## What to build next

Good next upgrades:
1. Add more medication rules
2. Add age-based risk scoring
3. Add Beers-style medication checks
4. Add symptom timeline logic
5. Add a simple web form with Gradio or Streamlit
6. Add an LLM summary layer for clearer explanations

Important safety note:
This tool is only for medication safety support.
It does **not** diagnose disease or tell a person to stop medication.
